In [1]:
import os, json, time, math, ast, random
from typing import Dict, Any, List, Tuple
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI, APIError, APIConnectionError, RateLimitError

In [2]:
load_dotenv()

# 모델/프롬프트 파라미터
MODEL = "gpt-4o-mini"
TEMPERATURE = 0.55
TOP_P = 0.9
MAX_TOKENS = 1400

# 병렬/오버제너레이션 파라미터 (LLM 페르소나 생성용)
POOL_SIZE   = 80
MIN_UNIQUE  = 25
MAX_WORKERS = 6
N_PERSONAS  = 20

# 예측 병렬 파라미터
FORECAST_BACKEND = "process"
FORECAST_MAX_WORKERS = os.cpu_count() or 4
FORECAST_CHUNK_SIZE = 10

# 파일 경로
INPUT_CSV = "data/product_info.csv"
OUT_PERSONAS_JSONL = "data/personas_20.jsonl"
OUT_FORECAST_CSV   = "data/forecast_by_sku_month.csv"

In [3]:
def call_llm_with_function(prompt: str) -> dict:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            tools=tools,
            tool_choice={"type": "function", "function": {"name": "create_persona_model"}},
            temperature=TEMPERATURE,
            top_p=TOP_P,
            max_tokens=MAX_TOKENS,
        )
        if response.choices and response.choices[0].message.tool_calls:
            output_str = response.choices[0].message.tool_calls[0].function.arguments
            return json.loads(output_str)
        else:
            print("[warn] LLM 응답에 유효한 함수 호출이 포함되지 않았습니다.")
            return None
    except APIConnectionError as e:
        print(f"[ERROR] 네트워크 오류: {e} (인터넷 연결/VPN/방화벽 확인)")
        return None
    except RateLimitError as e:
        print(f"[ERROR] 사용량 제한 오류: {e} (크레딧/플랜 확인)")
        return None
    except APIError as e:
        print(f"[ERROR] API 오류: {e} (API 키, 모델명 확인)")
        return None
    except json.JSONDecodeError:
        print(f"[ERROR] LLM 응답이 올바른 JSON 형식이 아닙니다.")
        return None
    except Exception as e:
        print(f"[ERROR] 알 수 없는 오류 발생: {e}")
        return None

In [4]:
def create_persona_model(persona: dict, purchase_model: dict):
    """
    OpenAI LLM 응답 스키마를 정의하는 함수.
    Args:
        persona (dict): 페르소나의 인적/심리적 정보.
        purchase_model (dict): 구매 행동 모델.
    """
    pass

tools = [{
    "type": "function",
    "function": {
        "name": "create_persona_model",
        "description": "JSON 객체를 생성하여 페르소나의 인적/심리적 정보와 구매 모델을 저장합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "persona": {
                    "type": "object",
                    "description": "페르소나의 인적/심리적 정보",
                    "properties": {
                        "name": {"type": "string"},
                        "gender": {"type": "string"},
                        "age": {"type": "integer"},
                        "occupation": {"type": "string"},
                        "key_traits": {"type": "array", "items": {"type": "string"}},
                        "main_channels": {"type": "array", "items": {"type": "string"}},
                        "budget_food_month_krw": {"type": "number"},
                        "flavor_preferences": {
                            "type": "object",
                            "properties": {
                                "spicy": {"type": "number"}, "sweet": {"type": "number"},
                                "savory": {"type": "number"}, "sour": {"type": "number"},
                                "umami": {"type": "number"},
                            },
                        },
                        "price_sensitivity": {"type": "number"},
                    },
                    "required": ["name", "gender", "age", "occupation", "key_traits", "main_channels", "budget_food_month_krw", "flavor_preferences", "price_sensitivity"],
                },
                "purchase_model": {
                    "type": "object",
                    "description": "페르소나의 구매 행동 모델",
                    "properties": {
                        "monthly_purchase_frequency": {"type": "array", "items": {"type": "number"}},
                        "avg_units_per_purchase": {"type": "number"},
                        "buy_probability": {"type": "number"},
                    },
                    "required": ["monthly_purchase_frequency", "avg_units_per_purchase", "buy_probability"],
                },
            },
            "required": ["persona", "purchase_model"],
        }
    }
}]

In [5]:
def build_prompt(product_for_prompt: Dict[str,Any], distinct_notes: str) -> str:
    prompt = f"""
    당신은 신제품 출시를 앞둔 마케팅 전문가입니다. 아래 제품을 구매할 잠재 고객 페르소나를 한 명 생성해야 합니다.
    
    제품 정보:
    {json.dumps(product_for_prompt, ensure_ascii=False, indent=2)}
    
    페르소나의 특징은 다음과 같습니다:
    {distinct_notes}
    """
    return prompt

def normalize_to_mean_one(weights, ndigits=3):
    if not weights or len(weights) != 12: raise ValueError("seasonality_weights_mean1 길이 12 아님")
    w = [max(0.0, float(x)) for x in weights]
    m = sum(w) / 12.0
    if m == 0: w = [1.0]*12
    else: w = [x / m for x in w]
    w = [round(x, ndigits) for x in w]
    diff = 12.0 - sum(w)
    if diff != 0:
        for i in range(len(w)): w[i] = round(w[i] + diff / len(w), ndigits)
    return w

def flavor_match(persona_flavors, product_flavors):
    if not persona_flavors or not product_flavors: return 0.5
    match_score = 0
    common_flavors = set(persona_flavors.keys()) & set(product_flavors.keys())
    if not common_flavors: return 0.5
    for flavor in common_flavors:
        score_diff = abs(persona_flavors[flavor] - product_flavors[flavor])
        match_score += (1 - score_diff)
    return match_score / len(common_flavors)

def greedy_maxmin_select(candidates, k=20, embedder=None):
    if len(candidates) <= k: return candidates
    if embedder is None: return random.sample(candidates, k)
    return random.sample(candidates, k)

def call_llm_with_function(prompt: str) -> dict:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    try:
        response = client.chat.completions.create(
            model=MODEL, messages=[{"role": "user", "content": prompt}], tools=tools,
            tool_choice={"type": "function", "function": {"name": "create_persona_model"}},
            temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_TOKENS,
        )
        if response.choices and response.choices[0].message.tool_calls:
            output_str = response.choices[0].message.tool_calls[0].function.arguments
            return json.loads(output_str)
        else:
            print("[warn] LLM 응답에 유효한 함수 호출이 포함되지 않았습니다.")
            return None
    except (APIConnectionError, RateLimitError, APIError, json.JSONDecodeError) as e:
        print(f"[ERROR] API 호출 중 오류 발생: {e}")
        return None
    except Exception as e:
        print(f"[ERROR] 알 수 없는 오류 발생: {e}")
        return None

def generate_persona_with_prompt(product_for_prompt, distinct_notes):
    try:
        prompt = build_prompt(product_for_prompt, distinct_notes=distinct_notes)
        return call_llm_with_function(prompt)
    except Exception as e:
        print(f"[warn] 페르소나 생성 중 예외 발생: {e}")
        return None

def row_to_product(row_series):
    prod = {
        "product_name": row_series["product_name"],
        "category": row_series["category"],
        "price": row_series["price"],
        "pack_info": row_series.get("pack_info", "기본"),
        "key_attributes": ast.literal_eval(row_series.get("key_attributes", "[]")),
        "channels": ast.literal_eval(row_series.get("channels", "[]")),
    }
    flavor_str = row_series.get("flavor_profile", "{}")
    prod["flavor_profile"] = ast.literal_eval(flavor_str)
    return prod

In [6]:
def forecast_sku_matrix(personas: list, product: dict, population_size: int = 51000000) -> pd.DataFrame:
    T = 12
    F = np.zeros((len(personas), T))
    scale = np.array([p.get('scale_factor', 1.0) for p in personas]).reshape(-1, 1)

    for i, persona in enumerate(personas):
        price_sensitivity = persona.get('persona', {}).get('price_sensitivity', 0.5)
        price_penalty = max(0, (product['price'] / persona.get('persona', {}).get('budget_food_month_krw', 1.0) - 0.005) * price_sensitivity * 10)
        
        flavor_score = flavor_match(persona.get('persona', {}).get('flavor_preferences', {}), product.get('flavor_profile', {}))
        
        persona_channels = set(c.strip().lower() for c in persona.get('persona', {}).get('main_channels', []))
        product_channels = set(c.strip().lower() for c in product.get('channels', []))
        channel_match_score = 1.0 if persona_channels.intersection(product_channels) else 0.5

        buy_probability = persona.get('purchase_model', {}).get('buy_probability', 0)
        adjusted_buy_probability = buy_probability * flavor_score * channel_match_score * (1 - price_penalty)
        
        monthly_purchase_frequency = persona.get('purchase_model', {}).get('monthly_purchase_frequency', [0]*12)
        avg_units_per_purchase = persona.get('purchase_model', {}).get('avg_units_per_purchase', 1)
        
        adjusted_monthly_frequency = [f * adjusted_buy_probability / buy_probability if buy_probability > 0 else 0 for f in monthly_purchase_frequency]
        
        F[i, :] = np.array(adjusted_monthly_frequency) * avg_units_per_purchase

    units = (scale * F).sum(axis=0)
    persona_count = len(personas)
    scaling_factor = population_size / persona_count if persona_count > 0 else 0

    price = product['price']

    scaled_units = units * scaling_factor
    revenue = scaled_units * price

    return pd.DataFrame({
        "month": np.arange(1, T + 1),
        "expected_units": scaled_units,
        "expected_revenue": revenue
    })

def generate_distinct_personas(product_for_prompt: Dict[str,Any], n:int=20) -> List[Dict[str,Any]]:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = []
        for _ in range(POOL_SIZE):
            distinct_notes = "독특한 관점과 구매 스타일을 가진 페르소나."
            futures.append(ex.submit(generate_persona_with_prompt, product_for_prompt, distinct_notes))
        
        candidates = []
        for future in as_completed(futures):
            try:
                persona_data = future.result()
                if persona_data: candidates.append(persona_data)
            except Exception as e: print(f"페르소나 생성 중 오류 발생: {e}")
                
    if not candidates:
        print("[warn] 최종 후보가 없습니다.")
        return []

    selected = greedy_maxmin_select(candidates, k=min(n, len(candidates)))

    total_w = sum(p.get("population_weight", 0.0) for p in selected) or 1.0

    for p in selected:
        p["population_weight"] = (p.get("population_weight", 0.0) / total_w) if total_w > 0 else 1.0 / len(selected)
        p['scale_factor'] = p['population_weight']

    if len(selected) < n: print(f"[warn] 후보가 부족해 {len(selected)}/{n}명만 선발했습니다.")
    return selected

def _run_parallel_forecast(df: pd.DataFrame, personas: List[Dict[str, Any]]) -> pd.DataFrame:
    if not personas: return pd.DataFrame(columns=["product_name","category","price","month","expected_units","expected_revenue"])
    records: List[Dict[str, Any]] = df.to_dict(orient="records")
    futures, results = [], []
    Executor = ProcessPoolExecutor if FORECAST_BACKEND == "process" else ThreadPoolExecutor
    if FORECAST_BACKEND != "process": print("[warn] 스레드 풀을 사용합니다. CPU 집약적 작업에는 ProcessPoolExecutor가 더 효율적일 수 있습니다.")
    with Executor(max_workers=FORECAST_MAX_WORKERS) as ex:
        for i in range(0, len(records), FORECAST_CHUNK_SIZE):
            chunk = records[i:i+FORECAST_CHUNK_SIZE]
            for row_dict in chunk:
                futures.append(ex.submit(_forecast_worker, row_dict, personas, 51000000))
    for future in as_completed(futures):
        try:
            res = future.result()
            if res is not None: results.append(res)
        except Exception as e: print(f"병렬 처리 중 오류 발생: {e}")
    if not results: return pd.DataFrame(columns=["product_name","category","price","price","month","expected_units","expected_revenue"])
    return pd.concat(results, ignore_index=True)

def _forecast_worker(row_dict: Dict[str, Any], personas: List[Dict[str, Any]], population_size: int) -> pd.DataFrame:
    row_series = pd.Series(row_dict)
    prod = row_to_product(row_series)
    agg = forecast_sku_matrix(personas, prod, population_size)
    agg["product_name"] = prod["product_name"]
    return agg

In [ ]:
def main():
    representative_product = {
        "product_name": "프리미엄 한우 육포", "category": "간식/주전부리", "price": 18000,
        "pack_info": "150g 스탠드 파우치", "key_attributes": ["고품질 한우", "전통방식 제조", "무첨가물", "선물용"],
        "channels": ["자사몰", "네이버 스마트스토어", "백화점 온라인몰"],
        "flavor_profile": {"spicy":0.1,"sweet":0.3,"savory":0.8,"sour":0.2,"umami":0.7}
    }
    
    print("--- 페르소나 생성 시작 ---")
    start_time = time.time()
    personas = generate_distinct_personas(representative_product, n=N_PERSONAS)
    end_time = time.time()
    print(f"총 페르소나 생성 시간: {end_time - start_time:.2f}초")

    if not personas:
        print("\n[fatal] 페르소나 생성 실패. 다음을 확인하세요:")
        print("1. **환경변수**: `.env` 파일에 `OPENAI_API_KEY`가 올바르게 설정되었는지 확인.")
        print(f"2. **모델명**: 사용 중인 모델 '{MODEL}'이 유효하고 접근 가능한지 확인.")
        print("3. **네트워크**: 인터넷 연결 상태나 방화벽 문제 확인.")
        print("4. **크레딧**: OpenAI 계정의 크레딧 잔액이 충분한지 확인.")
        return
    
    with open(OUT_PERSONAS_JSONL, "w", encoding="utf-8") as f:
        for p in personas: f.write(json.dumps(p, ensure_ascii=False) + "\n")

    try:
        df = pd.read_csv(INPUT_CSV)
    except FileNotFoundError:
        print(f"[fatal] 입력 파일 '{INPUT_CSV}'을 찾을 수 없습니다. 경로를 확인하세요.")
        return

    out_df = _run_parallel_forecast(df, personas)

    out_df['month_col'] = 'months_since_launch_' + out_df['month'].astype(str)
    pivot_df = out_df.pivot_table(
        index=['product_name'],
        columns='month_col',
        values='expected_units',
        fill_value=0
    ).reset_index()

    # 'sample_submission.csv' 형식과 일치하도록 컬럼 순서 재정렬
    desired_cols = ['product_name'] + [f'months_since_launch_{i}' for i in range(1, 13)]
    final_df = pivot_df.reindex(columns=desired_cols, fill_value=0)

    # 최종 CSV 저장
    final_df.to_csv(OUT_FORECAST_CSV, index=False, encoding="utf-8-sig")

    print(f"✅ SKU×월 예측 저장: {OUT_FORECAST_CSV}")
    print(final_df.head().to_string(index=False))

if __name__ == "__main__":
    main()

--- 페르소나 생성 시작 ---
총 페르소나 생성 시간: 51.15초
